In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, when, col, udf
from pyspark.sql.types import IntegerType, BooleanType, StringType
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re

spark = SparkSession.builder.appName("ETL Pipeline").getOrCreate()
STANDARD_SUBJECTS = [
    "News",
    "Politics",
    "World",
    "Business",
    "Technology",
    "Health",
    "Entertainment",
    "Sports",
    "Science",
    "Government",
]

# Function to standardize subjects
def standardize_subject(subject):
    if subject is None:
        return None
    subject = subject.strip().title()  # Normalize case and trim whitespace
    
    # First try exact matches
    if subject in STANDARD_SUBJECTS:
        return subject
    
    # Then try to map similar subjects to our standard ones
    subject_lower = subject.lower()
    
    if any(x in subject_lower for x in ["politics", "government", "election", "politicsNews"]):
        return "Politics"
    elif any(x in subject_lower for x in ["business", "economy", "finance", "market"]):
        return "Business"
    elif any(x in subject_lower for x in ["tech", "computer", "internet", "digital"]):
        return "Technology"
    elif any(x in subject_lower for x in ["health", "medical", "medicine", "hospital"]):
        return "Health"
    elif any(x in subject_lower for x in ["sports", "football", "basketball", "baseball"]):
        return "Sports"
    elif any(x in subject_lower for x in ["entertainment", "movie", "music", "celebrity"]):
        return "Entertainment"
    elif any(x in subject_lower for x in ["science", "research", "space", "physics"]):
        return "Science"
    elif any(x in subject_lower for x in ["world", "international", "global", "Middle-east", "worldnews"]):
        return "World"
    elif any(x in subject_lower for x in ["US_News", "left-news"]):
        return "News"

    # If no match found, default to "News" or None
    return None  # or return None if you want to drop non-matching subjects

standardize_subject_udf = udf(standardize_subject, StringType())


# Function to count words in a string
def count_words(s):
    if s is None:
        return 0
    return len(s.split())

# Function to check if string contains any numbers
def has_numbers(s):
    if s is None:
        return False
    return bool(re.search(r'\d', s))

# Register UDFs
count_words_udf = udf(count_words, IntegerType())
has_numbers_udf = udf(has_numbers, BooleanType())

# Fake news cleaned up
fake_df = spark.read.csv("/FileStore/tables/Fake.csv", header=True, inferSchema=True)
fake_df = fake_df.filter(~(col("title").isNull() & col("text").isNull() & col("subject").isNull() & col("date").isNull()))
fake_df_clean = (
    fake_df
    .withColumn("label", lit(0))  # 0 for fake
    .withColumn("subject", standardize_subject_udf(col("subject")))
    .filter(col("subject").isin(STANDARD_SUBJECTS))
    .select("title", "text", "subject", "date", "label")
)

# True news cleaned up
true_df = spark.read.csv("/FileStore/tables/True.csv", header=True, inferSchema=True)
true_df = true_df.filter(~(col("title").isNull() & col("text").isNull() & col("subject").isNull() & col("date").isNull()))
true_df_clean = (
    true_df
    .withColumn("subject", standardize_subject_udf(col("subject")))
    .filter(col("subject").isin(STANDARD_SUBJECTS))
    .withColumn("label", lit(1))  # 1 for real
    .select("title", "text", "subject", "date", "label")
)

# Both news csv cleaned up
both_df = spark.read.csv("/FileStore/tables/fake_news_both.csv", header=True, inferSchema=True)
both_df = both_df.filter(~(col("title").isNull() & col("text").isNull() & col("category").isNull() & col("date").isNull()))
both_df_clean = (
    both_df
    .drop("author", "source")
    .withColumnRenamed("category", "subject")
    .withColumn("subject", standardize_subject_udf(col("subject")))
    .filter(col("subject").isin(STANDARD_SUBJECTS))
    .withColumn("label", when(col("label") == "real", 1).when(col("label") == "fake", 0).otherwise(None))
    .select("title", "text", "subject", "date", "label")
)

fake_df2 = spark.read.csv("/FileStore/tables/DataSet_Misinfo_FAKE.csv", header=True, inferSchema=True)
fake_df2 = fake_df2.filter(~(col("text").isNull()))
fake_df_clean2 = (
    fake_df2
    .drop("index")
    .withColumn("label", lit(0))  # 0 for fake
    .select("text", "label")
)

true_df2 = spark.read.csv("/FileStore/tables/DataSet_Misinfo_TRUE.csv", header=True, inferSchema=True)
true_df2 = true_df2.filter(~(col("text").isNull()))
true_df_clean2 = (
    true_df2
    .drop("index")
    .withColumn("label", lit(1))  # 1 for real
    .select("text", "label")
)


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from datetime import datetime

def normalize_date(date_str):
    if date_str is None:
        return None
    formats = [
        "%b %d, %Y",      # Mar 2, 2016
        "%B %d, %Y",      # March 15, 2016
        "%b %d %Y",       # Jul 30 2015
        "%B %d %Y",       # December 15 2017
        "%Y-%m-%d",       # 2024-05-14
        "%d-%b-%y",       # 28-Jan-16, 1-Sep-17
        "%d-%b-%Y",       # 28-Jan-2016 (if present)
        "%m/%d/%y",       # 12/25/23
        "%m/%d/%Y",       # 12/25/2023
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str.strip(), fmt).strftime("%Y-%m-%d")
        except:
            continue
    return None  # Could not parse

normalize_date_udf = udf(normalize_date, StringType())


In [0]:
fake_df_clean = fake_df_clean.withColumn("date", normalize_date_udf(col("date")))
true_df_clean = true_df_clean.withColumn("date", normalize_date_udf(col("date")))

In [0]:
from pyspark.sql.functions import concat_ws, col, lit, rand, md5, concat, when
from pyspark.sql.types import StringType, IntegerType

# 1. Standardize and clean the secondary DataFrames (fake_df_clean2 and true_df_clean2)
fake_df_clean2 = fake_df_clean2.withColumn("title", lit(None).cast(StringType())) \
                              .withColumn("date", lit(None).cast(StringType())) \
                              .withColumn("subject", lit(None).cast(StringType())) \
                              .withColumn("combined_text", col("text")) \
                              .withColumn("label", col("label").cast(IntegerType())) \
                              .filter(col("label").isNotNull())  # Critical NULL filter

true_df_clean2 = true_df_clean2.withColumn("title", lit(None).cast(StringType())) \
                              .withColumn("date", lit(None).cast(StringType())) \
                              .withColumn("subject", lit(None).cast(StringType())) \
                              .withColumn("combined_text", col("text")) \
                              .withColumn("label", col("label").cast(IntegerType())) \
                              .filter(col("label").isNotNull())  # Critical NULL filter

# 2. Standardize the main DataFrames with NULL handling AND consistent casting
def standardize_df(df):
    return df.withColumn(
        "combined_text",
        concat_ws(" ", col("title"), col("date"), col("text"))
    ).withColumn(
        "label",
        when(col("label").isin(["0", "1"]), col("label").cast(IntegerType()))
        .when(col("label").isin([0, 1]), col("label").cast(IntegerType()))
        .otherwise(None)
    ).filter(col("label").isNotNull())  # Remove NULLs early

fake_df_clean = standardize_df(fake_df_clean)
true_df_clean = standardize_df(true_df_clean)
both_df_clean = standardize_df(both_df_clean)

# 3. PRIORITIZED SAMPLING APPROACH: Main sources first, then supplement
print("=== Prioritized Sampling Approach ===")

# Combine and deduplicate main sources (priority sources)
main_sources = fake_df_clean.unionByName(true_df_clean, allowMissingColumns=True) \
                           .unionByName(both_df_clean, allowMissingColumns=True)

print(f"Total main sources: {main_sources.count()}")

# Combine and deduplicate clean2 sources (supplemental sources)  
clean2_sources = fake_df_clean2.unionByName(true_df_clean2, allowMissingColumns=True)

print(f"Total clean2 sources: {clean2_sources.count()}")

# Deduplicate each group separately first
def deduplicate_group(df):
    df_typed = df.withColumn("label", col("label").cast(IntegerType()))
    return df_typed.withColumn(
        "content_hash", 
        md5(concat(col("combined_text")))
    ).dropDuplicates(["content_hash"]) \
     .drop("content_hash") \
     .filter(col("label").isNotNull())

main_dedup = deduplicate_group(main_sources)
clean2_dedup = deduplicate_group(clean2_sources)

print(f"Main sources after dedup: {main_dedup.count()}")
print(f"Clean2 sources after dedup: {clean2_dedup.count()}")

# Check available data in main sources
main_fake = main_dedup.filter(col("label") == 0)
main_real = main_dedup.filter(col("label") == 1)
clean2_fake = clean2_dedup.filter(col("label") == 0)
clean2_real = clean2_dedup.filter(col("label") == 1)

main_fake_count = main_fake.count()
main_real_count = main_real.count()
clean2_fake_count = clean2_fake.count()
clean2_real_count = clean2_real.count()

print(f"=== Available Data by Source ===")
print(f"Main fake: {main_fake_count}")
print(f"Main real: {main_real_count}")
print(f"Clean2 fake: {clean2_fake_count}")
print(f"Clean2 real: {clean2_real_count}")

# 4. Prioritized sampling strategy
sample_size = 100000
target_per_class = sample_size // 2  # 75,000 each

# Take ALL available from main sources first
fake_sample = main_fake
real_sample = main_real

fake_from_main = fake_sample.count()
real_from_main = real_sample.count()

print(f"=== Priority Sampling Results ===")
print(f"Fake from main: {fake_from_main}")
print(f"Real from main: {real_from_main}")

# Calculate how much more we need from clean2
needed_fake = max(0, target_per_class - fake_from_main)
needed_real = max(0, target_per_class - real_from_main)

print(f"Still need fake: {needed_fake}")
print(f"Still need real: {needed_real}")

# Supplement from clean2 sources only if needed
if needed_fake > 0:
    supplemental_fake = clean2_fake.orderBy(rand()).limit(min(needed_fake, clean2_fake_count))
    fake_sample = fake_sample.unionByName(supplemental_fake, allowMissingColumns=True)
    print(f"Added {supplemental_fake.count()} fake samples from clean2")

if needed_real > 0:
    supplemental_real = clean2_real.orderBy(rand()).limit(min(needed_real, clean2_real_count))
    real_sample = real_sample.unionByName(supplemental_real, allowMissingColumns=True)
    print(f"Added {supplemental_real.count()} real samples from clean2")

fake_sampled = fake_sample.count()
real_sampled = real_sample.count()

print(f"=== Final Sample Counts ===")
print(f"Total fake sampled: {fake_sampled}")
print(f"Total real sampled: {real_sampled}")

# 5. Combine final samples and remove any cross-source duplicates
final_dataset = fake_sample.unionByName(real_sample, allowMissingColumns=True)

print(f"Before final deduplication: {final_dataset.count()}")

# Final deduplication to remove any cross-source duplicates
final_dedup = final_dataset.withColumn(
    "content_hash", 
    md5(concat(col("combined_text")))
).dropDuplicates(["content_hash"]) \
 .drop("content_hash") \
 .filter(col("label").isNotNull())

print(f"After final deduplication: {final_dedup.count()}")

# Shuffle the final dataset
shuffled_df = final_dedup.orderBy(rand())

print(f"Final shuffled count: {shuffled_df.count()}")

# 6. Verify final distribution
print("=== Final Dataset Verification ===")
shuffled_df.groupBy("label").count().show()

# 7. Save with validation
shuffled_df.write.mode("overwrite").parquet("/FileStore/fake_news/cleaned")

final_df = spark.read.parquet("/FileStore/fake_news/cleaned") \
               .withColumn("label", col("label").cast(IntegerType())) \
               .filter(col("label").isNotNull())

print("=== Final Schema ===")
final_df.printSchema()

# Save to Delta
final_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("cleaned_news_data")

print("=== Final Validation ===")
print("Total rows:", final_df.count())
final_df.groupBy("label").count().show()
print("NULL labels:", final_df.filter(col("label").isNull()).count())

In [0]:
from pyspark.sql.functions import concat_ws, col, lit, rand, md5, concat, when
from pyspark.sql.types import StringType, IntegerType
final_df = spark.read.parquet("/FileStore/fake_news/cleaned") \
               .withColumn("label", col("label").cast(IntegerType())) \
               .filter(col("label").isNotNull())

In [0]:


train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

train_pd = train_df.select("combined_text", "label").toPandas()
test_pd = test_df.select("combined_text", "label").toPandas()

# Drop rows with nulls in 'text' or 'label'
train_pd = train_pd.dropna(subset=["combined_text", "label"])
test_pd = test_pd.dropna(subset=["combined_text", "label"])

# Convert all text entries to string type explicitly
train_pd["combined_text"] = train_pd["combined_text"].astype(str)
test_pd["combined_text"] = test_pd["combined_text"].astype(str)
train_pd["label"].value_counts()



In [0]:
import urllib.request
import gzip
import shutil

# Download URL
url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz"

# Local paths
gz_path = "/tmp/cc.en.300.bin.gz"
bin_path = "/tmp/cc.en.300.bin"

# Download to /tmp/
urllib.request.urlretrieve(url, gz_path)

# Extract the .bin file
with gzip.open(gz_path, 'rb') as f_in:
    with open(bin_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)


In [0]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix
import numpy as np
import pickle
import pandas as pd
from collections import defaultdict
import fasttext
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, accuracy_score, roc_auc_score, recall_score
import joblib
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
from hybrid_vectorizer import PrecomputedEmbeddingVectorizer

# Updated PyFunc Model for MLflow
class FakeNewsClassifierPyFunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import joblib
        # Load the pipeline (no FastText model needed!)
        self.pipeline = joblib.load(context.artifacts["pipeline_model"])
        # Load precomputed embeddings
        self.pipeline.named_steps["hybrid_features"].load_embeddings(context.artifacts["embeddings"])

    def predict(self, context, model_input):
        if isinstance(model_input, pd.DataFrame):
            texts = model_input['combined_text'].tolist() if 'combined_text' in model_input.columns else model_input.iloc[:, 0].tolist()
        else:
            texts = model_input if isinstance(model_input, list) else [model_input]
        
        preds = self.pipeline.predict(texts)
        probs = self.pipeline.predict_proba(texts)
        return {"predictions": preds.tolist(), "confidence_scores": probs.tolist()}



# Simple oversampling function (from your original code)
def simple_oversample(df, target_ratio=0.95):
    """Simple oversampling by duplicating minority class samples"""
    class_counts = df['label'].value_counts().sort_index()
    print(f"Before oversampling: {dict(class_counts)}")

    minority_class = class_counts.idxmin()
    majority_count = class_counts.max()
    minority_count = class_counts.min()

    print(f"Minority class: {minority_class}, count: {minority_count}")
    print(f"Majority count: {majority_count}")

    target_minority_count = int(majority_count * target_ratio)
    needed_samples = max(0, target_minority_count - minority_count)

    print(f"Target minority count: {target_minority_count}")
    print(f"Needed additional samples: {needed_samples}")

    if needed_samples > 0:
        minority_data = df[df['label'] == minority_class]
        additional_samples = minority_data.sample(n=needed_samples, replace=True, random_state=42)
        result_df = pd.concat([df, additional_samples], ignore_index=True)
        print(f"After oversampling: {dict(result_df['label'].value_counts().sort_index())}")
        return result_df

    print("No oversampling needed - classes already balanced enough")
    return df

# Direct replacement for your existing pipeline:

print("Original class distribution:")
print(train_pd["label"].value_counts())
print(f"Class ratio: {train_pd['label'].value_counts().min() / train_pd['label'].value_counts().max():.3f}")

# Apply oversampling (your existing code)
print("\nApplying simple oversampling...")
train_balanced = simple_oversample(train_pd, target_ratio=0.98)
print("Final balanced class distribution:")
print(train_balanced["label"].value_counts().sort_index())

train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Initialize vectorizers with precomputed embeddings
tfidf_vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words='english', lowercase=True)
hybrid_vec = PrecomputedEmbeddingVectorizer(tfidf_vec)

# Fit TF-IDF and precompute FastText embeddings
hybrid_vec.fit(train_balanced["combined_text"])
hybrid_vec.precompute_embeddings_from_fasttext("/tmp/cc.en.300.bin")  # Your FastText model path

# Save embeddings for serving
embeddings_path = "precomputed_embeddings.pkl"
hybrid_vec.save_embeddings(embeddings_path)

# Create and train pipeline
pipeline_model = Pipeline([
    ("hybrid_features", hybrid_vec),
    ("clf", VotingClassifier([
        ('rf', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42, class_weight='balanced')),
        ('lr', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
    ], voting='soft'))
])

pipeline_model.fit(train_balanced["combined_text"], train_balanced["label"])
test_predictions = pipeline_model.predict(test_pd["combined_text"])
test_probabilities = pipeline_model.predict_proba(test_pd["combined_text"])

print("\nClassification Report:")
print(classification_report(test_pd["label"], test_predictions))

# Save pipeline
joblib.dump(pipeline_model, "pipeline_model.pkl")

# MLflow logging (same position as your original code)
sample_input = pd.DataFrame({'text': ["Example text for testing"]})
sample_output = {"predictions": [1], "confidence_scores": [[0.2, 0.8]]}
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run(run_name="PrecomputedEmbeddingHybridModel"):
    test_f1 = f1_score(test_pd["label"], test_predictions, average='weighted')
    test_precision = precision_score(test_pd["label"], test_predictions, average='weighted')
    test_accuracy = accuracy_score(test_pd["label"], test_predictions)
    test_recall = recall_score(test_pd["label"], test_predictions, average='weighted')
    test_auc = roc_auc_score(test_pd["label"], test_probabilities[:, 1], multi_class='ovr')

    mlflow.log_metrics({
        "test_f1": test_f1,
        "test_precision": test_precision,
        "test_accuracy": test_accuracy,
        "test_recall": test_recall,
        "test_auc": test_auc
    })
    
    print("\nTest Set Metrics:")
    print(f"F1 Score (weighted): {test_f1:.4f}")
    print(f"Precision (weighted): {test_precision:.4f}")
    print(f"Accuracy: {test_accuracy:.4f}")
    print(f"Recall (weighted): {test_recall:.4f}")
    print(f"AUC (OvR): {test_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(test_pd["label"], test_predictions))

    mlflow.pyfunc.log_model(
        artifact_path="hybrid_fakenews_model",
        python_model=FakeNewsClassifierPyFunc(),
        artifacts={
            "pipeline_model": "pipeline_model.pkl",
            "embeddings": embeddings_path
        },
        code_path=["/Workspace/Users/aluu31@gatech.edu/hybrid_vectorizer.py"],
        signature=signature,
        input_example=sample_input,
        registered_model_name="FakeNewsDataBoostedMemoryEfficient",
        pip_requirements=['scikit-learn==1.3.0', 'scipy', 'numpy', 'pandas','fasttext==0.9.2']
    )

In [0]:
# from hybrid_vectorizer import HybridVectorizer


In [0]:
# import urllib.request
# import gzip
# import shutil
# import os
# import fasttext
# import pandas as pd
# import numpy as np
# from sklearn.base import BaseEstimator, TransformerMixin
# from scipy.sparse import hstack, csr_matrix
# from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier, VotingClassifier
# from sklearn.metrics import classification_report, f1_score, precision_score, accuracy_score, roc_auc_score, recall_score
# import joblib
# import mlflow
# import mlflow.pyfunc
# from mlflow.models.signature import infer_signature


# print("Original class distribution:")
# print(train_pd["label"].value_counts())
# print(f"Class ratio: {train_pd['label'].value_counts().min() / train_pd['label'].value_counts().max():.3f}")

# # Simple oversampling function (alternative to SMOTE)
# def simple_oversample(df, target_ratio=0.95):
#     """Simple oversampling by duplicating minority class samples"""
#     class_counts = df['label'].value_counts().sort_index()
#     print(f"Before oversampling: {dict(class_counts)}")
    
#     minority_class = class_counts.idxmin()
#     majority_count = class_counts.max()
#     minority_count = class_counts.min()
    
#     print(f"Minority class: {minority_class}, count: {minority_count}")
#     print(f"Majority count: {majority_count}")
    
#     target_minority_count = int(majority_count * target_ratio)
#     needed_samples = max(0, target_minority_count - minority_count)
    
#     print(f"Target minority count: {target_minority_count}")
#     print(f"Needed additional samples: {needed_samples}")
    
#     if needed_samples > 0:
#         minority_data = df[df['label'] == minority_class]
#         additional_samples = minority_data.sample(n=needed_samples, replace=True, random_state=42)
#         result_df = pd.concat([df, additional_samples], ignore_index=True)
#         print(f"After oversampling: {dict(result_df['label'].value_counts().sort_index())}")
#         return result_df
    
#     print("No oversampling needed - classes already balanced enough")
#     return df


# print("\nApplying simple oversampling...")
# train_balanced = simple_oversample(train_pd, target_ratio=0.98)
# print("Final balanced class distribution:")
# print(train_balanced["label"].value_counts().sort_index())


# train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
# tfidf_vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words='english', lowercase=True)
# hybrid_vec = HybridVectorizer(tfidf_vec)
# hybrid_vec.load_fasttext_model(bin_path)

# pipeline_model = Pipeline([
#     ("hybrid_features", hybrid_vec),
#     ("clf", VotingClassifier([
#         ('rf', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42, class_weight='balanced')),
#         ('lr', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
#     ], voting='soft'))
# ])

# pipeline_model.fit(train_balanced["combined_text"], train_balanced["label"])


In [0]:
# test_predictions = pipeline_model.predict(test_pd["combined_text"])
# test_probabilities = pipeline_model.predict_proba(test_pd["combined_text"])

# print("\nClassification Report:")
# print(classification_report(test_pd["label"], test_predictions))


# # === 5. Save pipeline ===
# pipeline_model.named_steps["hybrid_features"].ft_model = None  # remove unpicklable object
# joblib.dump(pipeline_model, "pipeline_model.pkl")


# # === 7. Define PyFunc Model ===
# class FakeNewsClassifierPyFunc(mlflow.pyfunc.PythonModel):
#     def load_context(self, context):
#         import joblib
#         import fasttext
#         self.pipeline = joblib.load(context.artifacts["pipeline_model"])
#         self.pipeline.named_steps["hybrid_features"].load_fasttext_model(context.artifacts["ft_model"])

#     def predict(self, context, model_input):
#         if isinstance(model_input, pd.DataFrame):
#             texts = model_input['combined_text'].tolist() if 'combined_text' in model_input.columns else model_input.iloc[:, 0].tolist()
#         else:
#             texts = model_input if isinstance(model_input, list) else [model_input]
#         preds = self.pipeline.predict(texts)
#         probs = self.pipeline.predict_proba(texts)
#         return {"predictions": preds.tolist(), "confidence_scores": probs.tolist()}

# # === 8. Log with MLflow ===
# sample_input = pd.DataFrame({'text': ["Example text for testing"]})
# sample_output = {"predictions": [1], "confidence_scores": [[0.2, 0.8]]}
# signature = infer_signature(sample_input, sample_output)

# with mlflow.start_run(run_name="FastTextHybridModel"):
#     test_f1 = f1_score(test_pd["label"], test_predictions, average='weighted')
#     test_precision = precision_score(test_pd["label"], test_predictions, average='weighted')
#     test_accuracy = accuracy_score(test_pd["label"], test_predictions)
#     test_recall = recall_score(test_pd["label"], test_predictions, average='weighted')
#     test_auc = roc_auc_score(test_pd["label"], test_probabilities[:, 1], multi_class='ovr')

#     mlflow.log_metrics({
#         "test_f1": f1_score(test_pd["label"], test_predictions, average='weighted'),
#         "test_precision": precision_score(test_pd["label"], test_predictions, average='weighted'),
#         "test_accuracy": accuracy_score(test_pd["label"], test_predictions),
#         "test_recall": recall_score(test_pd["label"], test_predictions, average='weighted'),
#         "test_auc": roc_auc_score(test_pd["label"], test_probabilities[:, 1], multi_class='ovr')
#     })
#     print("\nTest Set Metrics:")
#     print(f"F1 Score (weighted): {test_f1:.4f}")
#     print(f"Precision (weighted): {test_precision:.4f}")
#     print(f"Accuracy: {test_accuracy:.4f}")
#     print(f"Recall (weighted): {test_recall:.4f}")
#     print(f"AUC (OvR): {test_auc:.4f}")
    
#     print("\nClassification Report:")
#     print(classification_report(test_pd["label"], test_predictions))
    
#     mlflow.pyfunc.log_model(
#         artifact_path="hybrid_fakenews_model",
#         python_model=FakeNewsClassifierPyFunc(),
#         artifacts={
#             "pipeline_model": "pipeline_model.pkl",
#             "ft_model": "/tmp/cc.en.50.bin"
#         },
#         code_path=["/Workspace/Users/aluu31@gatech.edu/hybrid_vectorizer.py"],
#         signature=signature,
#         input_example=sample_input,
#         registered_model_name="FakeNewsDataBoosted",
#         pip_requirements=['fasttext==0.9.2', 'scikit-learn==1.3.0']
#     )    

In [0]:
import mlflow
logged_model = 'runs:/e72bc523ecd04a0fa5c034d81ad9ea57/hybrid_fakenews_model'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)
data = "When it comes to ending the war in Ukraine, President Donald Trump’s statements and social media posts have become meaningless. Receding chances for a ceasefire and peace deal soon will depend instead on whether he finally finds the steel to reinforce his rhetorical lashing of President Vladimir Putin over the weekend with action. The Kremlin is betting he won’t. It dismissed Trump’s frustration with the most intense Russian drone attacks on Ukraine as a symptom of “emotional overload.” And experience suggests Putin can get away with calling the US president’s bluff. After all, Trump’s Truth Social critique of the Russian leader as “crazy” on Monday was leavened with a characteristic rebuke of the victim — Ukraine and President Volodymyr Zelensky. Still, the intensifying Russian attacks on Ukrainian civilians appear to be a deliberate Russian test for Trump, a week after his hyped call with Putin, which made no progress toward peace despite the White House spin. There are two routes Trump can take, assuming he’s ready to abandon the embarrassing position of being constantly played by Putin. He could impose new sanctions against Russia, which he previously argued would hamper diplomacy. He told reporters in New Jersey on Sunday this was “absolutely” a consideration. Trump could also save lives in Ukraine by emulating his predecessor Joe Biden and asking Congress to approve new shipments of arms and ammunition to the country. But this option would mean a massive turnaround that would be embarrassing politically, since Trump’s opposition to spending billions of dollars in Ukraine is a foundation of his second presidency. And it would mean the president accepting that, as was the case for many of his recent predecessors, his belief that he could manage Putin was flawed. There is another possibility — one that Ukraine and its European allies fear. Trump could throw up his hands and argue that neither side wants peace and it’s time for the US to walk away. Russia would then press on with its war of attrition and attacks on civilians. Its land grab would be validated, creating a disastrous precedent for European security and US disengagement. This isn’t an academic prospect. An isolationist streak running through the MAGA movement meant that recent hints by Secretary of State Marco Rubio and Vice President JD Vance that the US could step back seem like more than a mere negotiating tactic. And one way to read Trump’s Truth Social post on Monday was as a smokescreen for a US withdrawal. “This is a War that would never have started if I were President. This is Zelenskyy’s, Putin’s, and Biden’s War, not ‘Trump’s,’” the president wrote."


In [0]:
# Predict on a Pandas DataFrame.
import pandas as pd
loaded_model.predict(pd.DataFrame({'text': [data]}))

In [0]:
# Create a debug version
import mlflow
import pandas as pd

# Load the model
loaded_model = mlflow.pyfunc.load_model(logged_model)

# Try to access the underlying sklearn model
sklearn_model = loaded_model._model_impl.sklearn_model

# Compare predictions
df_input = pd.DataFrame({'text': [data]})
print("DataFrame input:", df_input)

# Test direct sklearn model
direct_prediction = sklearn_model.predict([data])
print("Direct sklearn prediction:", direct_prediction)

# Test PyFunc
pyfunc_prediction = loaded_model.predict(df_input)
print("PyFunc prediction:", pyfunc_prediction)

# Check if text is being extracted correctly
extracted_text = df_input['text'].iloc[0]
print("Extracted text (first 100 chars):", extracted_text[:100])

In [0]:
import mlflow.sklearn

# Load as sklearn model (not PyFunc)
sklearn_model = mlflow.sklearn.load_model(logged_model)

# Predict like local (works!)
print(sklearn_model.predict(["The vaccine causes microchips in humans."]))  # Should return 0
print(sklearn_model.predict(["Hong Kong/Beijing CNN — At least five people have died and six more people are missing after a large explosion rocked a chemical plant in eastern China on Tuesday, according to local authorities. The blast spewed a towering plume of gray and orange smoke into the sky, damaged windows in nearby buildings and prompted a rescue operation. The incident happened in the workshop of Shandong Youdao Chemical in Gaomi city, Shandong province, minutes before noon local time, state broadcaster CCTV reported. It did not give a reason for the explosion. A further 19 people suffered minor injuries, local authorities said in a statement. Videos circulating on Chinese social media showed smoke engulfing buildings in an industrial park. Windows in some nearby low-rise buildings were damaged, the footage showed. The local fire and rescue services dispatched 55 vehicles and 232 personnel to the scene, while the Ministry of Emergency Management dispatched a working group and rescue reinforcements, the ministry said in a statement. A staff member working at a hotel some 3.5 kilometers (2.2 miles) from the blast site said she heard the explosion around noon. “The sound was quite loud with a bang. It only lasted for a moment,” she told CNN, adding that the hotel did not suffer any damage. Another worker at a factory about 6 kilometers (3.7 miles) from the blast site said she heard a boom and felt a shake and a “strong gust” of wind. “A strong gust of airflow scared me so much that I didn’t dare leave my office,” said the worker, surnamed Meng. “The doors and windows in (my) factory were damaged… The airflow rushed in through the window, and if I had been a bit closer, it might have thrown me against the wall.” Shandong Youdao Chemical is owned by Himile Group, which also owns listed Himile Mechanical, shares of which were down nearly 4% on Tuesday afternoon, according to Reuters. Founded in August 2019, Shandong Youdao Chemical occupies more than 46 hectares of land in the Gaomi Renhe chemical park and employs more than 300 people, according to the company’s website. It develops, produces and sells pesticides, pharmaceuticals and chemical intermediates, the website says. In 2015 a series of blasts at a chemical warehouse in the northeastern city of Tianjin killed more than 100 people and sent toxic fumes into the air."]))  # Should return 1

In [0]:
# Test different input formats
test_inputs = [
    pd.Series([data]),  # Series format (matches your training signature)
    [data],             # List format
    pd.DataFrame([data]), # DataFrame without column name
    pd.DataFrame({'text': [data]}) # Your current format
]

for i, test_input in enumerate(test_inputs):
    try:
        result = loaded_model.predict(test_input)
        print(f"Input format {i} works: {result}")
    except Exception as e:
        print(f"Input format {i} failed: {str(e)}")

In [0]:
print(pipeline_model.predict(["The vaccine causes microchips in humans."]))
# Should return 0 (fake), not 1

print(pipeline_model.predict(["Hong Kong/Beijing CNN — At least five people have died and six more people are missing after a large explosion rocked a chemical plant in eastern China on Tuesday, according to local authorities. The blast spewed a towering plume of gray and orange smoke into the sky, damaged windows in nearby buildings and prompted a rescue operation. The incident happened in the workshop of Shandong Youdao Chemical in Gaomi city, Shandong province, minutes before noon local time, state broadcaster CCTV reported. It did not give a reason for the explosion. A further 19 people suffered minor injuries, local authorities said in a statement. Videos circulating on Chinese social media showed smoke engulfing buildings in an industrial park. Windows in some nearby low-rise buildings were damaged, the footage showed. The local fire and rescue services dispatched 55 vehicles and 232 personnel to the scene, while the Ministry of Emergency Management dispatched a working group and rescue reinforcements, the ministry said in a statement. A staff member working at a hotel some 3.5 kilometers (2.2 miles) from the blast site said she heard the explosion around noon. “The sound was quite loud with a bang. It only lasted for a moment,” she told CNN, adding that the hotel did not suffer any damage. Another worker at a factory about 6 kilometers (3.7 miles) from the blast site said she heard a boom and felt a shake and a “strong gust” of wind. “A strong gust of airflow scared me so much that I didn’t dare leave my office,” said the worker, surnamed Meng. “The doors and windows in (my) factory were damaged… The airflow rushed in through the window, and if I had been a bit closer, it might have thrown me against the wall.” Shandong Youdao Chemical is owned by Himile Group, which also owns listed Himile Mechanical, shares of which were down nearly 4% on Tuesday afternoon, according to Reuters. Founded in August 2019, Shandong Youdao Chemical occupies more than 46 hectares of land in the Gaomi Renhe chemical park and employs more than 300 people, according to the company’s website. It develops, produces and sells pesticides, pharmaceuticals and chemical intermediates, the website says. In 2015 a series of blasts at a chemical warehouse in the northeastern city of Tianjin killed more than 100 people and sent toxic fumes into the air."]))
# Should return 1 (real)


In [0]:
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]
top_fake = sorted(zip(coefs, feature_names))[:10]
top_real = sorted(zip(coefs, feature_names))[-10:]
print("Top fake words:", top_fake)
print("Top real words:", top_real)




In [0]:
dbutils.fs.mkdirs("/FileStore/tables/fake_news_stream")
##Starting here and below is the real time streaming

In [0]:
%sql

CREATE OR REPLACE TABLE fake_news_streaming (
  title STRING,
  text STRING,
  date STRING,
  label INT,
  combined_text STRING
)
USING delta

In [0]:
from pyspark.sql.functions import col

original_data = spark.table("cleaned_news_data")

# Select and cast columns in the correct order and types
original_data = original_data.select(
    col("title").cast("string"),
    col("text").cast("string"),
    col("subject").cast("string"),
    col("date").cast("string"),
    col("label").cast("int"),
    col("combined_text").cast("string")
)

original_data.write.mode("overwrite").format("delta").saveAsTable("added_data")


In [0]:
new_batch = spark.read.json("/FileStore/tables/fake_news_stream/log1.json")

new_batch = new_batch.select(
    col("title").cast("string"),
    col("text").cast("string"),
    col("subject").cast("string"),
    col("date").cast("string"),
    col("label").cast("int"),
    col("combined_text").cast("string")
)

new_batch.write.mode("append").format("delta").saveAsTable("added_data")